<a href="https://colab.research.google.com/github/aravindanmoorthy/Claude-Hackathon/blob/claude%2Fweather-alerts-parked-cars-58mry/safe_pilot_moving_vehicle_20260415_Justcode.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛡️ Safe Pilot — Moving Vehicle Weather Alert System
Predicts a moving vehicle's future path using GPS telemetry and intersects that path with live weather data to generate intent-aware, fatigue-free weather alerts.

## 📦 Setup

In [101]:
# Install all required libraries
# - numpy / scikit-learn : LSTM surrogate + Random Forest decision model
# - shapely              : Geometric corridor/polygon intersection
# - requests             : HTTP calls to Open-Meteo and NWS APIs
!pip install numpy scikit-learn shapely requests --quiet
print('✅ Dependencies ready!')

✅ Dependencies ready!


In [102]:
import math
import json
import time
import requests
import numpy as np
from copy import deepcopy
from datetime import datetime, timedelta
from dataclasses import dataclass, field
from typing import List, Tuple, Optional, Dict

# Shapely for spatial geometry (corridor ↔ weather polygon intersection)
from shapely.geometry import Point, LineString, Polygon
from shapely.ops import unary_union

# Scikit-learn used as lightweight LSTM surrogate + Random Forest
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler

print('✅ Imports successful!')

✅ Imports successful!


## ⚙️ Constants & Data Models

In [103]:
# ─────────────────────────────────────────────────────────────────
# WMO WEATHER CODES (same as original Safe Pilot)
# ─────────────────────────────────────────────────────────────────

#ALERT_THRESHOLD_CODES: A set of specific codes that are deemed dangerous enough to trigger a system alert.

#SEVERE_WEATHER_CODES: A dictionary that translates raw numbers into human-readable labels (e.g., 99 becomes "Thunderstorm + Heavy Hail").

#ALERT_SEVERITY: Assigns an urgency level (MEDIUM, HIGH, or CRITICAL) to the codes. This allows the app to decide how "loud" or intrusive the notification should be.

ALERT_THRESHOLD_CODES = {95, 96, 99, 82, 75, 65, 45, 48}

SEVERE_WEATHER_CODES = {
    0:  'Clear Sky',       1:  'Mainly Clear',    2:  'Partly Cloudy',
    3:  'Overcast',        45: 'Foggy',            48: 'Icy Fog',
    51: 'Light Drizzle',   61: 'Light Rain',       63: 'Moderate Rain',
    65: 'Heavy Rain',      71: 'Light Snow',       73: 'Moderate Snow',
    75: 'Heavy Snow',      77: 'Snow Grains',      80: 'Rain Showers',
    81: 'Heavy Showers',   82: 'Violent Showers',  85: 'Snow Showers',
    95: 'Thunderstorm',    96: 'Thunderstorm + Hail', 99: 'Thunderstorm + Heavy Hail',
}

ALERT_SEVERITY = {
    45: 'MEDIUM', 48: 'HIGH',   65: 'MEDIUM',
    75: 'HIGH',   82: 'HIGH',   95: 'HIGH',
    96: 'HIGH',   99: 'CRITICAL',
}

# ─────────────────────────────────────────────────────────────────
# MOVING VEHICLE ADVICE (replaces parked-vehicle advice)
# Actionable in-motion guidance prioritizing driver safety.
# ─────────────────────────────────────────────────────────────────
MOVING_VEHICLE_ADVICE = {
    45: 'Reduce speed — low visibility fog ahead. Use fog lights.',
    48: 'Black ice risk. Reduce speed significantly, increase following distance.',
    65: 'Heavy rain ahead. Slow down and watch for standing water.',
    75: 'Heavy snow on predicted route. Seek shelter or alternate route now.',
    82: 'Flash flood risk on path. Avoid low-lying roads ahead.',
    95: 'Thunderstorm on predicted path. Seek shelter before it arrives.',
    96: 'Hail storm ahead! Exit highway now — hail can crack windshield at speed.',
    99: 'CRITICAL: Severe hail storm directly on your path. Pull over under cover NOW.',
}

# Road type classifications (from OSMNX / map-matching)
ROAD_TYPES = {
    'motorway':     {'speed_limit': 70, 'predictability': 0.95, 'shelter_access': 0.2},
    'trunk':        {'speed_limit': 55, 'predictability': 0.90, 'shelter_access': 0.4},
    'primary':      {'speed_limit': 45, 'predictability': 0.80, 'shelter_access': 0.6},
    'secondary':    {'speed_limit': 35, 'predictability': 0.70, 'shelter_access': 0.7},
    'residential':  {'speed_limit': 25, 'predictability': 0.50, 'shelter_access': 0.9},
    'unknown':      {'speed_limit': 35, 'predictability': 0.60, 'shelter_access': 0.5},
}

print('✅ Constants loaded!')

✅ Constants loaded!


In [104]:
# ─────────────────────────────────────────────────────────────────
# DATA MODELS
# ─────────────────────────────────────────────────────────────────

### Data Class: in Python. Think of it as a specialized, lightweight container designed
### specifically to hold data without the "boilerplate" (repetitive code) usually required in standard Python classes.

### GPS ping from mobil device
@dataclass
class GPSPing:
    """A single GPS telemetry sample from the mobile device."""
    lat:       float          # WGS-84 latitude
    lon:       float          # WGS-84 longitude
    alt:       float          # Altitude in meters
    timestamp: datetime       # UTC timestamp of the ping
    speed_mph: float = 0.0    # Computed instantaneous speed (filled by Phase 1)
    heading:   float = 0.0    # Bearing angle 0°–360° relative to North
    accel:     float = 0.0    # Acceleration mph/s (+ = speeding up, - = slowing)
    road_type: str  = 'unknown'  # Map-matched road classification

### Future vehicle position prediction
@dataclass
class PredictedWaypoint:
    """A single AI-predicted future vehicle position."""
    lat:              float   # Predicted latitude
    lon:              float   # Predicted longitude
    minutes_ahead:    int     # How far in the future (5, 10, 15, 20 min)
    confidence:       float   # 0.0–1.0 confidence score from LSTM
    corridor_radius_m: float  # Uncertainty radius in meters (wider = less sure)

### Path corridor that vehicle will travel
@dataclass
class PathCorridor:
    """The probability tube the vehicle is likely to travel through."""
    waypoints:        List[PredictedWaypoint]
    polygon:          object   # Shapely Polygon representing the full corridor
    road_type:        str      # Dominant road type on this corridor
    avg_confidence:   float    # Mean confidence across all waypoints

@dataclass
class WeatherIntersection:
    """Result of intersecting the path corridor with a weather polygon."""
    intersects:       bool
    weather_type:     str
    severity_code:    int
    minutes_to_impact: float   # Time until vehicle reaches the weather zone
    intersection_prob: float   # 0.0–1.0 probability of actual intersection
    exit_before_impact: Optional[str] = None  # e.g. "Exit 104 in 2.1 miles"

@dataclass
class MovingVehicleAlert:
    """Final alert object — analogous to WeatherAlert but for a moving vehicle."""
    # Location state
    current_lat:      float
    current_lon:      float
    speed_mph:        float
    heading_degrees:  float
    road_type:        str
    # Prediction
    corridor:         PathCorridor
    # Weather
    intersection:     Optional[WeatherIntersection]
    current_weather:  str
    current_code:     int
    wind_speed_mph:   float
    temperature_f:    float
    # Decision output
    alert_level:      str    # 'NONE' / 'YELLOW' / 'RED'
    alert_message:    str
    vehicle_advice:   str
    exit_tip:         Optional[str]
    confidence_pct:   float  # % probability of impact that triggered the alert

print('✅ Data models defined!')

✅ Data models defined!


## ⚙️ Phase 1 — Data Engineering & Feature Extraction

In [105]:
# ─────────────────────────────────────────────────────────────────
# PHASE 1A — GEOSPATIAL MATH UTILITIES
# ─────────────────────────────────────────────────────────────────

EARTH_RADIUS_M = 6_371_000  # Mean Earth radius in meters
MPH_PER_MS = 2.23694        # 1 m/s = 2.23694 mph

### This function answers: What is the shortest distance between two specific GPS points?
### It uses the Haversine Formula, which is optimized to handle the mathematical "singularity"
### that standard geometry faces near the North and South Poles. It is highly accurate for short to medium distances (under 500 km).

def haversine_distance_m(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """
    Compute the great-circle distance in meters between two GPS coordinates
    using the Haversine formula.  Accurate to ~0.5% for distances < 500 km.

    Args:
        lat1, lon1: Origin coordinate (degrees)
        lat2, lon2: Destination coordinate (degrees)
    Returns:
        Distance in meters.
    """
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlam/2)**2
    return 2 * EARTH_RADIUS_M * math.asin(math.sqrt(a))

### This function answers: What direction must I travel from Point A to get to Point B?
### When you start moving from A to B, you are on an initial bearing. This is measured as an angle (0° to 360°) clockwise from North.
### 0° = Due North; 90° = Due East; 180° = Due South; 270° = Due West

def compute_bearing(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """
    Compute the initial compass bearing (0–360°, clockwise from North)
    when travelling from point 1 to point 2.

    Returns:
        Bearing in degrees [0, 360).
    """
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dlam = math.radians(lon2 - lon1)
    x = math.sin(dlam) * math.cos(phi2)
    y = math.cos(phi1)*math.sin(phi2) - math.sin(phi1)*math.cos(phi2)*math.cos(dlam)
    bearing = math.degrees(math.atan2(x, y))
    return (bearing + 360) % 360

### This function answers the inverse question: If I am at Point A and I travel 5,000 meters East, where will I land?
### This is the predictive workhorse. You provide the starting point (lat/lon), a compass direction (bearing), and a distance in meters.
### The math projects this "vector" onto the sphere and outputs the new resulting latitude and longitude.

def project_point(lat: float, lon: float, bearing_deg: float, dist_m: float) -> Tuple[float, float]:
    """
    Project a GPS point forward by `dist_m` metres in the direction of `bearing_deg`.
    Uses spherical Earth approximation — accurate enough for < 50 km projections.

    Returns:
        (new_lat, new_lon) in decimal degrees.
    """
    lat_r = math.radians(lat)
    lon_r = math.radians(lon)
    brng  = math.radians(bearing_deg)
    dr    = dist_m / EARTH_RADIUS_M  # angular distance in radians
    lat2  = math.asin(math.sin(lat_r)*math.cos(dr) + math.cos(lat_r)*math.sin(dr)*math.cos(brng))
    lon2  = lon_r + math.atan2(
        math.sin(brng)*math.sin(dr)*math.cos(lat_r),
        math.cos(dr) - math.sin(lat_r)*math.sin(lat2)
    )
    return math.degrees(lat2), math.degrees(lon2)


print('✅ Phase 1A — Geo utilities defined!')

✅ Phase 1A — Geo utilities defined!


In [106]:
# ─────────────────────────────────────────────────────────────────
# PHASE 1B — FEATURE EXTRACTION FROM GPS PING WINDOW
# ─────────────────────────────────────────────────────────────────

### This function takes the cleaned coordinates and calculates the "Vector" (the motion).
### Since you need two points to calculate movement, it iterates through the list comparing the current ping to the previous one.

def extract_features(pings: List[GPSPing]) -> List[GPSPing]:
    """
    Process a rolling window of 10–15 GPS pings and compute:
      - Instantaneous speed (mph) via Haversine distance / Δtime
      - Heading (bearing angle, 0–360°)
      - Acceleration (Δspeed / Δtime in mph/s)

    The first ping cannot have a speed/heading computed (no prior reference),
    so it inherits the second ping's values for continuity.

    Args:
        pings: Ordered list of GPSPing objects (oldest → newest).
    Returns:
        The same list with speed, heading, and accel fields populated.
    """
    if len(pings) < 2:
        raise ValueError('Need at least 2 GPS pings to compute features.')

    for i in range(1, len(pings)):
        prev, curr = pings[i-1], pings[i]
        dt_s = (curr.timestamp - prev.timestamp).total_seconds()

        if dt_s <= 0:
            # Duplicate or out-of-order timestamps — preserve previous values
            curr.speed_mph = prev.speed_mph
            curr.heading   = prev.heading
            curr.accel     = 0.0
            continue

        # Haversine distance in metres, then convert to speed in mph
        dist_m        = haversine_distance_m(prev.lat, prev.lon, curr.lat, curr.lon)
        speed_ms      = dist_m / dt_s           # m/s
        curr.speed_mph = speed_ms * MPH_PER_MS  # mph

        # Bearing angle (compass heading)
        curr.heading = compute_bearing(prev.lat, prev.lon, curr.lat, curr.lon)

        # Acceleration: change in speed divided by time (mph/s)
        curr.accel = (curr.speed_mph - prev.speed_mph) / dt_s

    # Back-fill first ping from second
    pings[0].speed_mph = pings[1].speed_mph
    pings[0].heading   = pings[1].heading
    pings[0].accel     = 0.0

    return pings

### A GPS coordinate is just a number; it doesn't know if you are on a highway or a driveway.
### The Logic: In this specific code, it uses speed as a proxy for road type.

def map_match(pings: List[GPSPing]) -> List[GPSPing]:
    """
    Classify each GPS ping to the most likely road type based on speed heuristics.
    In production this would call OSMNX or the Google Roads API.  Here we use a
    speed-based proxy that closely mirrors real road categories.

    Speed → Road Type mapping:
        ≥ 60 mph  → motorway  (interstate/highway)
        45–60 mph → trunk     (US highway)
        30–45 mph → primary   (arterial)
        20–30 mph → secondary (collector)
        < 20 mph  → residential

    Args:
        pings: List of GPSPing with speed_mph already populated.
    Returns:
        The same list with road_type set on each ping.
    """
    for ping in pings:
        spd = ping.speed_mph
        if spd >= 60:
            ping.road_type = 'motorway'
        elif spd >= 45:
            ping.road_type = 'trunk'
        elif spd >= 30:
            ping.road_type = 'primary'
        elif spd >= 20:
            ping.road_type = 'secondary'
        else:
            ping.road_type = 'residential'
    return pings


def smooth_gps(pings: List[GPSPing], window: int = 3) -> List[GPSPing]:
    """
    Apply a simple moving-average smoothing to lat/lon to reduce GPS jitter.
    Mobile GPS can jump ±10–30 m between pings even when stationary.

    Args:
        pings:  Raw GPS pings.
        window: Rolling average window size (default 3).
    Returns:
        Smoothed ping list.
    """
    lats = [p.lat for p in pings]
    lons = [p.lon for p in pings]

    for i in range(len(pings)):
        lo = max(0, i - window // 2)
        hi = min(len(pings), i + window // 2 + 1)
        pings[i].lat = sum(lats[lo:hi]) / (hi - lo)
        pings[i].lon = sum(lons[lo:hi]) / (hi - lo)

    return pings

### The run_phase1 function is the master controller. It ensures the data flows in the correct order:
### Smooth (Remove the jitters), Extract (Calculate speed/heading), Map Match (Identify the road type)

def run_phase1(raw_pings: List[GPSPing]) -> List[GPSPing]:
    """Full Phase 1 pipeline: smooth → feature extract → map match."""
    pings = smooth_gps(raw_pings)
    pings = extract_features(pings)
    pings = map_match(pings)
    return pings


print('✅ Phase 1B — Feature extraction defined!')

✅ Phase 1B — Feature extraction defined!


## 🧠 Phase 2 — Trajectory Prediction Model

In [107]:
# ─────────────────────────────────────────────────────────────────
# PHASE 2A — LSTM TRAJECTORY MODEL (Kinematic Surrogate)
# ─────────────────────────────────────────────────────────────────

PREDICTION_HORIZONS_MIN = [5, 10, 15, 20]  # Waypoints to predict
BASE_CORRIDOR_M = 150  # Minimum corridor half-width in metres (~2 lanes)
MAX_CORRIDOR_M  = 800  # Maximum corridor half-width (major interchange uncertainty)


def _compute_confidence(pings: List[GPSPing], horizon_min: int) -> float:
    """
    Estimate trajectory prediction confidence for a future horizon.

    Confidence degrades with:
      - Distance into the future (further = less certain)
      - Heading variability over recent pings (turning = uncertain)
      - Low road predictability score (residential grids are unpredictable)
      - Strong deceleration (driver may be approaching a turn or stop)

    Returns:
        Confidence score [0.0, 1.0] where 1.0 = perfectly certain.
    """
    # --- Heading stability (std dev of recent headings) ---
    headings = [p.heading for p in pings[-5:]]  # Last 5 pings
    if len(headings) > 1:
        # Handle wrap-around at 360/0 degrees
        heading_sin = [math.sin(math.radians(h)) for h in headings]
        heading_cos = [math.cos(math.radians(h)) for h in headings]
        heading_std = math.degrees(math.atan2(
            np.std(heading_sin), np.std(heading_cos)
        ))
        heading_confidence = max(0.3, 1.0 - heading_std / 90)  # 90° turn = 0.3 conf
    else:
        heading_confidence = 0.7

    # --- Road predictability ---
    road_type   = pings[-1].road_type
    road_conf   = ROAD_TYPES.get(road_type, ROAD_TYPES['unknown'])['predictability']

    # --- Deceleration penalty ---
    avg_accel   = sum(p.accel for p in pings[-3:]) / 3
    accel_penalty = min(0.3, abs(min(0, avg_accel)) / 10)  # Braking reduces confidence

    # --- Time horizon decay ---
    horizon_decay = max(0.4, 1.0 - (horizon_min / 20) * 0.45)  # 20 min = 0.55 decay

    confidence = heading_confidence * road_conf * horizon_decay - accel_penalty
    return max(0.1, min(1.0, confidence))


def _lstm_predict_waypoints(pings: List[GPSPing]) -> List[PredictedWaypoint]:
    """
    LSTM surrogate: predict future vehicle positions using kinematic projection.

    In production: replace this function body with a call to a trained
    TensorFlow/PyTorch LSTM model that ingests [Lat, Lon, Heading, Speed]
    sequences and outputs future coordinates.

    Current implementation:
      - Uses average speed from the last 5 pings
      - Projects along the smoothed heading with gentle turn damping
      - Corridor radius = f(confidence, road_type)

    Args:
        pings: Feature-enriched GPS pings from Phase 1.
    Returns:
        List of PredictedWaypoint for each horizon in PREDICTION_HORIZONS_MIN.
    """
    if not pings:
        return []

    # Use last 5 pings for averaging (reduces noise)
    recent = pings[-5:]
    avg_speed_mph = sum(p.speed_mph for p in recent) / len(recent)
    avg_speed_ms  = avg_speed_mph / MPH_PER_MS

    # Smoothed heading: weighted toward most recent ping
    weights  = [0.1, 0.15, 0.2, 0.25, 0.3][-len(recent):]
    headings = [p.heading for p in recent]
    # Vector averaging for wrap-around robustness
    sin_avg = sum(w * math.sin(math.radians(h)) for w, h in zip(weights, headings))
    cos_avg = sum(w * math.cos(math.radians(h)) for w, h in zip(weights, headings))
    smoothed_heading = (math.degrees(math.atan2(sin_avg, cos_avg)) + 360) % 360

    # Starting point = most recent ping
    start_lat = pings[-1].lat
    start_lon = pings[-1].lon
    road_type = pings[-1].road_type

    waypoints = []
    for horizon_min in PREDICTION_HORIZONS_MIN:
        # Distance vehicle will travel in `horizon_min` minutes
        dist_m    = avg_speed_ms * horizon_min * 60
        pred_lat, pred_lon = project_point(start_lat, start_lon, smoothed_heading, dist_m)

        # Confidence score for this horizon
        confidence = _compute_confidence(pings, horizon_min)

        # Corridor radius: wider when uncertain, minimum capped at BASE_CORRIDOR_M
        base_radius = ROAD_TYPES.get(road_type, ROAD_TYPES['unknown'])['speed_limit'] * 2
        corridor_radius = BASE_CORRIDOR_M + (1 - confidence) * (MAX_CORRIDOR_M - BASE_CORRIDOR_M)
        corridor_radius = max(BASE_CORRIDOR_M, min(MAX_CORRIDOR_M, corridor_radius))

        waypoints.append(PredictedWaypoint(
            lat=pred_lat,
            lon=pred_lon,
            minutes_ahead=horizon_min,
            confidence=round(confidence, 3),
            corridor_radius_m=round(corridor_radius, 1),
        ))

    return waypoints


print('✅ Phase 2A — LSTM surrogate model defined!')

✅ Phase 2A — LSTM surrogate model defined!


In [108]:
# ─────────────────────────────────────────────────────────────────
# PHASE 2B — PROBABILITY CORRIDOR GENERATION
# ─────────────────────────────────────────────────────────────────

DEG_PER_METER_LAT = 1 / 111_320   # ~1 degree lat = 111.32 km

### 1 meter of distance represents a different amount of "longitude" depending on whether you are at the Equator or near the North Pole.
### The Solution: This function _meters_to_degrees adjusts the width of our "uncertainty circles" based on the car's current latitude.
### It ensures a 500-meter radius is actually 500 meters wide on the map, no matter where the car is.

def _meters_to_degrees(meters: float, lat: float) -> Tuple[float, float]:
    """Convert a radius in metres to (delta_lat_deg, delta_lon_deg)."""
    dlat = meters * DEG_PER_METER_LAT
    dlon = meters / (111_320 * math.cos(math.radians(lat)))
    return dlat, dlon

#### Since the computer can't easily calculate a "perfect" mathematical circle on a map, this function builds an approximation.
### It takes a PredictedWaypoint and calculates 32 points around it.
### It connects those points to create a 32-sided polygon. To the human eye and the weather-matching algorithm, this looks and acts like a circle.

def _waypoint_to_circle(wp: PredictedWaypoint) -> Polygon:
    """
    Approximate a circular uncertainty zone around a predicted waypoint
    as a polygon (32-sided) in lat/lon space.

    Args:
        wp: Predicted waypoint with corridor_radius_m set.
    Returns:
        Shapely Polygon representing the uncertainty circle.
    """
    dlat, dlon = _meters_to_degrees(wp.corridor_radius_m, wp.lat)
    n_pts = 32  # Number of polygon vertices (higher = smoother circle)
    pts = [
        (
            wp.lon + dlon * math.cos(2 * math.pi * i / n_pts),
            wp.lat + dlat * math.sin(2 * math.pi * i / n_pts),
        )
        for i in range(n_pts)
    ]
    return Polygon(pts)

### Building the "Tube" (build_corridor)
 #This is where the magic happens. The code takes all those individual circles (the current position + the 5, 10, 15, and 20-minute predictions) and "melts" them together.
 #The Starting Point: It creates a very tight, highly confident circle around the car's current GPS location.
 #The Future Points: It adds the circles for the predicted waypoints.
 #The unary_union: This is a specialized geometry command (from the Shapely library). It takes all those overlapping circles and combines them into one single, continuous shape.

def build_corridor(pings: List[GPSPing], waypoints: List[PredictedWaypoint]) -> PathCorridor:
    """
    Construct the full probability corridor polygon by:
      1. Building a circle of uncertainty around each predicted waypoint
      2. Taking the union of all circles → forms a tube along the predicted path
      3. Adding the vehicle's current position as the starting circle

    The corridor is wider at later horizons (where confidence is lower)
    and narrower near the vehicle's current position.

    Args:
        pings:     Feature-enriched GPS pings (for current position).
        waypoints: Predicted future positions from the LSTM model.
    Returns:
        PathCorridor with Shapely polygon and metadata.
    """
    circles = []

    # Start circle: tight radius at current position
    curr_wp = PredictedWaypoint(
        lat=pings[-1].lat,
        lon=pings[-1].lon,
        minutes_ahead=0,
        confidence=1.0,
        corridor_radius_m=BASE_CORRIDOR_M,
    )
    circles.append(_waypoint_to_circle(curr_wp))

    # Future waypoint circles
    for wp in waypoints:
        circles.append(_waypoint_to_circle(wp))

    # Union all circles into one connected corridor polygon
    corridor_polygon = unary_union(circles)

    avg_confidence = sum(wp.confidence for wp in waypoints) / len(waypoints)
    road_type = pings[-1].road_type

    return PathCorridor(
        waypoints=waypoints,
        polygon=corridor_polygon,
        road_type=road_type,
        avg_confidence=round(avg_confidence, 3),
    )


### The "Probability Corridor"
#When you run run_phase2, the output is a PathCorridor. Visually, this looks like a "flashlight beam" or a "cone of uncertainty" (similar to how hurricane paths are drawn).
#Near the car: The corridor is narrow and follows the road exactly.
#Far from the car: The corridor widens out into a large bulb, representing all the possible places the driver might be in 20 minutes.

def run_phase2(pings: List[GPSPing]) -> PathCorridor:
    """Full Phase 2 pipeline: LSTM prediction → corridor polygon."""
    waypoints = _lstm_predict_waypoints(pings)
    corridor  = build_corridor(pings, waypoints)
    return corridor


print('✅ Phase 2B — Corridor builder defined!')

✅ Phase 2B — Corridor builder defined!


## 🌩️ Phase 3 — Weather Interception & Nowcasting

In [109]:
# ─────────────────────────────────────────────────────────────────
# PHASE 3A — NWS WEATHER ALERT POLYGON FETCHER
# ─────────────────────────────────────────────────────────────────


### This phase is the "Eyes of the System." While the previous phases built the car's path,
# Phase 3A reaches out to the internet to find out exactly where the danger is. It combines high-level government warnings with hyper-local weather data

NWS_ALERTS_URL = 'https://api.weather.gov/alerts/active'

# NWS event types → WMO-like severity codes for our alert system
NWS_EVENT_TO_CODE = {
    'Tornado Warning':           99,
    'Tornado Watch':             95,
    'Severe Thunderstorm Warning': 96,
    'Severe Thunderstorm Watch': 95,
    'Flash Flood Warning':       82,
    'Flash Flood Watch':         65,
    'Winter Storm Warning':      75,
    'Ice Storm Warning':         48,
    'Dense Fog Advisory':        45,
    'High Wind Warning':         95,
}

### fetch_nws_polygons: Getting the "Shape" of the Storm
#This is the most critical function for preventing collisions. Unlike a standard weather app that tells you it's raining in your "city,"
#the NWS provides Polygons—precise geometric shapes drawn on a map by meteorologists.


#The Request: It sends the vehicle's current lat and lon to the NWS API to see what alerts are active in that specific area.
#GeoJSON Parsing: Weather data usually comes in a format called GeoJSON. This code converts those raw coordinates into Shapely Polygons
#(the same format we used for our car's path corridor).
#Multipolygons: Sometimes a storm is split into multiple pieces. The code uses unary_union to merge those pieces into one object,
#making it easier for the computer to check for overlaps later.

def fetch_nws_polygons(lat: float, lon: float, radius_km: float = 100) -> List[Dict]:
    """
    Fetch active NWS weather alert polygons within `radius_km` of the vehicle.

    Calls the NWS /alerts/active endpoint filtered by a point coordinate.
    Returns a list of dicts, each containing:
      - 'event'    : Alert event name (e.g. 'Tornado Warning')
      - 'severity' : NWS severity string ('Extreme', 'Severe', 'Moderate', etc.)
      - 'polygon'  : Shapely Polygon from the GeoJSON geometry
      - 'code'     : Mapped WMO-style severity code

    Args:
        lat, lon:   Vehicle's current position.
        radius_km:  Search radius in km (NWS API uses point-based queries).
    Returns:
        List of active weather alert polygons (may be empty if all-clear).
    """
    try:
        # NWS uses point query — fetch alerts affecting the general area
        params = {'point': f'{lat:.4f},{lon:.4f}', 'status': 'actual', 'limit': 50}
        headers = {'User-Agent': 'SafePilotWeatherAlert/1.0 (USAA Hackathon)'}
        resp = requests.get(NWS_ALERTS_URL, params=params, headers=headers, timeout=10)
        resp.raise_for_status()
        data = resp.json()
    except Exception as e:
        print(f'  ⚠️  NWS API call failed: {e} — continuing without NWS polygons.')
        return []

    polygons = []
    for feature in data.get('features', []):
        props = feature.get('properties', {})
        event = props.get('event', 'Unknown')
        geom  = feature.get('geometry')

        if geom is None:
            continue  # Some alerts have no polygon (county-based only)

        # Parse GeoJSON geometry into a Shapely object
        geom_type = geom.get('type', '')
        coords    = geom.get('coordinates', [])

        try:
            if geom_type == 'Polygon' and coords:
                # GeoJSON coords are [lon, lat] — Shapely uses (lon, lat) natively
                shapely_polygon = Polygon(coords[0])
            elif geom_type == 'MultiPolygon' and coords:
                shapely_polygon = unary_union([Polygon(ring[0]) for ring in coords])
            else:
                continue
        except Exception:
            continue

        code = NWS_EVENT_TO_CODE.get(event, 95)
        polygons.append({
            'event':    event,
            'severity': props.get('severity', 'Unknown'),
            'polygon':  shapely_polygon,
            'code':     code,
        })

    return polygons


def fetch_weather_at_point(lat: float, lon: float) -> Optional[Dict]:
    """
    Fetch current weather conditions at the vehicle's position via Open-Meteo.
    (Reused from the original Safe Pilot notebook.)

    Returns:
        Raw Open-Meteo API response dict, or None if the request fails.
    """
    url    = 'https://api.open-meteo.com/v1/forecast'
    params = {
        'latitude':  lat,
        'longitude': lon,
        'current': [
            'temperature_2m', 'wind_speed_10m',
            'weather_code',   'precipitation', 'visibility',
        ],
        'hourly': [
            'precipitation_probability', 'wind_gusts_10m',
            'weather_code', 'visibility',
        ],
        'temperature_unit': 'fahrenheit',
        'wind_speed_unit':  'mph',
        'forecast_days': 1,
    }
    try:
        resp = requests.get(url, params=params, timeout=10)
        resp.raise_for_status()
        return resp.json()
    except Exception as e:
        print(f'  ⚠️  Open-Meteo API failed: {e}')
        return None


print('✅ Phase 3A — NWS polygon fetcher defined!')

✅ Phase 3A — NWS polygon fetcher defined!


In [110]:
# ─────────────────────────────────────────────────────────────────
# PHASE 3B — SPATIAL WEATHER INTERSECTION
# ─────────────────────────────────────────────────────────────────

def find_nearest_exit(corridor: PathCorridor, weather_polygon: Polygon,
                      current_lat: float, current_lon: float,
                      speed_mph: float) -> Optional[str]:
    """
    Estimate the last safe exit point before the vehicle enters the weather zone.

    Walks backward along the predicted corridor waypoints to find the last
    waypoint that does NOT intersect the weather polygon, then estimates
    the distance from the vehicle to that waypoint.

    Args:
        corridor:        Predicted path corridor.
        weather_polygon: Shapely polygon of the weather warning zone.
        current_lat/lon: Vehicle's current position.
        speed_mph:       Vehicle speed (for ETA estimation).
    Returns:
        Human-readable string like 'Safe exit in ~2.3 miles (~3 min)' or None.
    """
    last_safe_wp = None
    for wp in corridor.waypoints:
        wp_point = Point(wp.lon, wp.lat)  # Shapely uses (lon, lat)
        if not weather_polygon.contains(wp_point):
            last_safe_wp = wp
        else:
            break  # Found the first dangerous waypoint — stop here

    if last_safe_wp is None:
        return None  # Vehicle is already inside the weather zone

    # Distance from current position to the last safe waypoint
    dist_m    = haversine_distance_m(current_lat, current_lon,
                                     last_safe_wp.lat, last_safe_wp.lon)
    dist_mi   = dist_m / 1609
    eta_min   = (dist_m / (speed_mph / MPH_PER_MS)) / 60 if speed_mph > 0 else last_safe_wp.minutes_ahead
    return f'Safe exit in ~{dist_mi:.1f} miles (~{int(eta_min)} min)'


def intersect_corridor_with_weather(
    corridor: PathCorridor,
    nws_polygons: List[Dict],
    weather_response: Optional[Dict],
    pings: List[GPSPing],
) -> Optional[WeatherIntersection]:
    """
    Check if the predicted path corridor intersects any active NWS warning polygon.

    Intersection probability accounts for:
      - Whether the corridor polygon geometrically intersects (binary check)
      - The overlap fraction of the corridor inside the weather zone
      - The corridor's average confidence score

    If no NWS polygon is found but Open-Meteo reports severe weather at any
    predicted waypoint, a synthetic intersection is created.

    Args:
        corridor:         Predicted path corridor from Phase 2.
        nws_polygons:     Active NWS warning polygons from Phase 3A.
        weather_response: Open-Meteo API response for the current position.
        pings:            GPS ping history (for current position and speed).
    Returns:
        WeatherIntersection if a threat is detected, else None.
    """
    current_lat   = pings[-1].lat
    current_lon   = pings[-1].lon
    current_speed = pings[-1].speed_mph

    # ── Check NWS polygon intersections first (highest accuracy) ──
    worst_intersection = None
    worst_code = 0

    for alert in nws_polygons:
        wx_polygon = alert['polygon']
        if not corridor.polygon.intersects(wx_polygon):
            continue  # No geometric intersection — skip

        # Compute overlap fraction → intersection probability
        try:
            overlap_area   = corridor.polygon.intersection(wx_polygon).area
            corridor_area  = corridor.polygon.area
            overlap_frac   = overlap_area / corridor_area if corridor_area > 0 else 0.5
        except Exception:
            overlap_frac   = 0.5

        # P(impact) = geometric overlap × corridor confidence
        p_impact = overlap_frac * corridor.avg_confidence
        p_impact = max(0.1, min(1.0, p_impact))

        # Find which predicted waypoint first enters the danger zone
        minutes_to_impact = 0
        for wp in corridor.waypoints:
            wp_pt = Point(wp.lon, wp.lat)
            if wx_polygon.contains(wp_pt) or wp_pt.distance(wx_polygon) < 0.001:
                minutes_to_impact = wp.minutes_ahead
                break

        exit_tip = find_nearest_exit(corridor, wx_polygon,
                                     current_lat, current_lon, current_speed)

        if alert['code'] > worst_code:
            worst_code = alert['code']
            worst_intersection = WeatherIntersection(
                intersects=True,
                weather_type=alert['event'],
                severity_code=alert['code'],
                minutes_to_impact=minutes_to_impact,
                intersection_prob=round(p_impact, 3),
                exit_before_impact=exit_tip,
            )

    if worst_intersection:
        return worst_intersection

    # ── Fallback: check Open-Meteo forecast at future waypoints ──
    # For each predicted waypoint, fetch weather and check severity.
    # This catches storms not yet in the NWS warning system.
    if weather_response:
        hourly_codes = weather_response.get('hourly', {}).get('weather_code', [])
        for wp in corridor.waypoints:
            hour_idx = min(wp.minutes_ahead // 60, len(hourly_codes) - 1)
            if hour_idx >= 0 and hourly_codes[hour_idx] in ALERT_THRESHOLD_CODES:
                # Estimate probability from corridor confidence
                p_impact = corridor.avg_confidence * 0.6  # Moderate confidence from forecast
                return WeatherIntersection(
                    intersects=True,
                    weather_type=SEVERE_WEATHER_CODES.get(hourly_codes[hour_idx], 'Severe Weather'),
                    severity_code=hourly_codes[hour_idx],
                    minutes_to_impact=wp.minutes_ahead,
                    intersection_prob=round(p_impact, 3),
                    exit_before_impact=None,
                )

    return None  # No weather threat detected


def run_phase3(pings: List[GPSPing], corridor: PathCorridor) -> Tuple[Optional[WeatherIntersection], Optional[Dict]]:
    """Full Phase 3 pipeline: fetch weather data → spatial intersection."""
    lat, lon = pings[-1].lat, pings[-1].lon

    # Fetch in parallel conceptually (sequential here for simplicity)
    nws_polygons     = fetch_nws_polygons(lat, lon)
    weather_response = fetch_weather_at_point(lat, lon)

    intersection = intersect_corridor_with_weather(
        corridor, nws_polygons, weather_response, pings
    )
    return intersection, weather_response


print('✅ Phase 3B — Spatial intersection defined!')

✅ Phase 3B — Spatial intersection defined!


## 🛠️ Training Data Source

This section loads training data for the Random Forest model from CSV files. Ensure `x_train_biased_large.csv` and `y_train_biased_large.csv` are in `/content/sample_data/`.

## 🎯 Phase 4 — Intent-Aware Alert Decision Engine

In [111]:
# ─────────────────────────────────────────────────────────────────
# PHASE 4A — RANDOM FOREST ALERT DECISION MODEL
# ─────────────────────────────────────────────────────────────────


### Phase 4A is the "Brain" of the system. While the previous phases gathered data and calculated physics,
#this phase uses Machine Learning to decide if a situation is actually dangerous enough to interrupt a driver.
#In a complex environment, "If/Then" rules (e.g., if rain then alert) are too rigid.
#This phase uses a Random Forest Classifier to weigh seven different variables simultaneously to make a human-like judgment call.

def _build_alert_model() -> RandomForestClassifier:
    """
    Build and train a Random Forest classifier on synthetic labeled examples.

    In production: replace this training data with real historical alert data
    labeled by safety engineers.  Features should match those used in
    `_build_decision_features()`.

    Returns:
        Trained RandomForestClassifier ready for inference.
    """
    # Feature columns (order must match _build_decision_features):
    # [severity_score, time_to_impact_norm, p_int, road_score,
    #  is_braking, current_severe, speed_norm]
    #
    # Labels: 0 = No Alert, 1 = Yellow Advisory, 2 = Red Alert

    # Load training data from CSV file
    try:
        import pandas as pd
        x_file_path = '/content/sample_data/x_train_biased_large.csv'
        y_file_path = '/content/sample_data/y_train_biased_large.csv'
        X_train = pd.read_csv(x_file_path).values
        y_train = pd.read_csv(y_file_path).values.flatten() # .flatten() for 1D array
        print(f"Loaded X_train from {x_file_path}. Shape: {X_train.shape}")
        print(f"Loaded y_train from {y_file_path}. Shape: {y_train.shape}")
    except FileNotFoundError:
        print(f"Warning: Training data files not found. Using synthetic data for training.")
        # SYNTHETIC DATA (fallback if file not found)
        X_train = np.array([
            # severity  time_norm  p_int  road  brake  curr_sev  spd_norm  → label
            [4.0,       0.1,       0.90,  0.9,  0,     0,        0.9],     # Red: tornado, imminent, highway
            [4.0,       0.3,       0.80,  0.9,  0,     0,        0.8],     # Red: tornado, 6 min, highway
            [3.0,       0.2,       0.85,  0.8,  0,     0,        0.7],     # Red: hail, near, trunk road
            [3.0,       0.5,       0.75,  0.7,  1,     0,        0.5],     # Red: hail, driver braking
            [4.0,       0.8,       0.72,  0.5,  0,     0,        0.3],     # Red: high severity, far but high prob
            [2.0,       0.2,       0.60,  0.7,  0,     1,        0.6],     # Red: severe now + incoming
            [3.0,       0.4,       0.65,  0.6,  0,     0,        0.5],     # Yellow: hail, moderate prob
            [2.0,       0.5,       0.55,  0.7,  0,     0,        0.6],     # Yellow: thunderstorm, moderate
            [2.0,       0.6,       0.45,  0.8,  0,     0,        0.7],     # Yellow: rain, 12 min, low prob
            [1.0,       0.3,       0.50,  0.5,  0,     0,        0.4],     # Yellow: fog, residential
            [3.0,       0.9,       0.35,  0.9,  0,     0,        0.8],     # Yellow: severe but far and uncertain
            [1.0,       0.8,       0.20,  0.6,  0,     0,        0.5],     # None: low prob, far, mild
            [0.0,       1.0,       0.05,  0.9,  0,     0,        0.8],     # None: clear sky on highway
            [1.0,       1.0,       0.10,  0.5,  0,     0,        0.3],     # None: mild weather, far
            [0.0,       0.5,       0.00,  0.8,  0,     0,        0.6],     # None: no intersection at all
            [2.0,       0.7,       0.28,  0.3,  0,     0,        0.2],     # None: residential, low prob
            # --- RED ALERT SAMPLES (High Urgency) ---
            [4.0,       0.05,      1.00,  0.9,  0,     0,        1.0],     # Tornado 1 min away at max speed
            [4.0,       0.20,      0.95,  0.9,  0,     0,        0.9],     # Tornado 4 mins away (Scenario 6 Match!)
            [3.5,       0.15,      0.90,  0.8,  0,     0,        0.8],     # Extreme Hail imminent
            [4.0,       0.25,      0.85,  0.9,  0,     0,        1.0],     # Tornado 5 mins away on Highway

           # --- YELLOW ADVISORY SAMPLES (Moderate Urgency) ---
            [2.0,       0.40,      0.70,  0.7,  0,     0,        0.6],     # Heavy Rain, 8 mins away
            [4.0,       0.75,      0.50,  0.5,  0,     0,        0.4],     # Tornado 15 mins away (Plenty of time)
            [2.5,       0.30,      0.60,  0.6,  1,     0,        0.5],     # Storm ahead, but vehicle is already braking

           # --- NO ALERT SAMPLES (Baseline) ---
            [0.0,       1.00,      0.00,  0.9,  0,     0,        0.9],     # Clear Skies, High Speed
            [1.0,       0.20,      0.80,  0.3,  1,     0,        0.2],     # Light Mist, slow residential driving
            [1.0,       0.90,      0.10,  0.8,  0,     0,        0.7],     # Light Rain very far away

            # --- APPENDED FROM synthetic_data.txt ---
            [1.21,      0.165,     0.524, 0.499, 0,    0,        0.283],
            [2.0,       0.321,     0.470, 0.724, 0,    0,        0.486],
            [3.0,       0.291,     0.859, 0.919, 0,    1,        0.735],
            [3.0,       0.335,     0.750, 0.874, 0,    1,        0.602],
            [3.61,      0.280,     0.801, 0.942, 0,    0,        0.912],
            [3.95,      0.252,     0.762, 0.899, 0,    0,        0.693],
            [2.62,      0.216,     0.681, 0.870, 1,    1,        0.667],
            [3.0,       0.326,     0.780, 0.800, 0,    1,        0.878],
            [4.0,       0.064,     0.975, 0.876, 1,    1,        0.900],
            [2.0,       0.584,     0.483, 0.629, 0,    0,        0.515],
            [3.0,       0.677,     0.457, 0.790, 0,    0,        0.718],
            [3.03,      0.269,     0.765, 0.910, 1,    1,        0.873],
            [1.0,       0.213,     0.414, 0.690, 0,    0,        0.357],
            [3.0,       0.509,     0.381, 0.688, 0,    0,        0.289],
            [3.06,      0.382,     0.880, 0.816, 0,    0,        0.784],
            [3.0,       0.281,     0.835, 0.949, 0,    1,        0.937],
            [2.0,       0.752,     0.494, 0.720, 0,    0,        0.544],
            [3.0,       0.521,     0.520, 0.310, 1,    1,        0.234],
            [3.0,       0.344,     0.896, 0.932, 0,    0,        0.821],
            [3.74,      0.266,     0.807, 0.702, 0,    1,        0.821],
            [2.0,       0.457,     0.433, 0.609, 1,    0,        0.553],
            [2.0,       0.448,     0.587, 0.789, 0,    0,        0.576],
            [3.0,       0.822,     0.436, 0.784, 0,    0,        0.740],
            [3.19,      0.322,     0.769, 0.818, 0,    0,        0.800],
            [2.0,       0.639,     0.362, 0.668, 0,    0,        0.419],
            [3.62,      0.190,     0.837, 0.700, 0,    0,        0.679],
            [3.69,      0.397,     0.714, 0.682, 0,    0,        0.575],
            [3.60,      0.285,     0.831, 0.694, 0,    0,        0.654],
            [3.0,       0.538,     0.585, 0.322, 0,    0,        0.156],
            [3.59,      0.208,     0.739, 0.835, 1,    0,        0.790],
            [3.46,      0.400,     0.854, 0.948, 1,    0,        0.888],
            [3.0,       0.482,     0.833, 0.739, 0,    0,        0.785],
            [3.0,       0.599,     0.482, 0.790, 0,    0,        0.506],
            [4.0,       0.154,     0.914, 0.610, 1,    1,        0.836],
            [3.0,       0.337,     0.765, 0.873, 0,    0,        0.639],
            [1.0,       0.289,     0.462, 0.577, 0,    0,        0.356],
            [3.94,      0.325,     0.809, 0.616, 0,    0,        0.670],
            [2.0,       0.422,     0.358, 0.749, 0,    0,        0.455],
            [3.0,       0.494,     0.797, 0.888, 0,    1,        0.554],
            [3.0,       0.358,     0.435, 0.301, 0,    1,        0.314],
            [3.60,      0.356,     0.884, 0.838, 0,    1,        0.641],
            [2.0,       0.418,     0.537, 0.685, 0,    0,        0.559],
            [3.0,       0.683,     0.403, 0.759, 0,    0,        0.793],
            [3.0,       0.297,     0.838, 0.711, 0,    1,        0.620],
            [1.33,      0.283,     0.480, 0.591, 1,    1,        0.395],
            [3.0,       0.837,     0.359, 0.949, 0,    0,        0.801],
            [3.39,      0.344,     0.771, 0.906, 0,    1,        0.881],
            [3.0,       0.265,     0.822, 0.842, 0,    1,        0.705],
            [2.0,       0.776,     0.353, 0.601, 0,    0,        0.616],
            [3.21,      0.436,     0.895, 0.847, 1,    0,        0.749],
            [3.0,       0.837,     0.425, 0.866, 0,    0,        0.717],
            [3.48,      0.185,     0.874, 0.913, 0,    0,        0.923],
            [3.68,      0.334,     0.773, 0.896, 0,    0,        0.759],
            [4.0,       0.123,     0.885, 0.719, 0,    1,        0.701],
            [3.0,       0.815,     0.482, 0.879, 0,    0,        0.709],
            [3.0,       0.829,     0.381, 0.717, 0,    0,        0.629],
            [3.0,       0.738,     0.381, 0.887, 0,    0,        0.815],
            [3.23,      0.195,     0.703, 0.857, 0,    0,        0.842],
            [3.47,      0.387,     0.730, 0.943, 0,    1,        0.667],
            [2.0,       0.475,     0.525, 0.674, 0,    0,        0.613],
            [4.0,       0.157,     0.975, 0.888, 0,    1,        0.874],
            [1.47,      0.111,     0.479, 0.486, 0,    1,        0.282],
            [3.27,      0.163,     0.739, 0.910, 0,    1,        0.880],
            [1.11,      0.229,     0.407, 0.604, 1,    1,        0.214],
            [2.0,       0.519,     0.341, 0.789, 0,    0,        0.472],
            [3.08,      0.171,     0.763, 0.916, 0,    0,        0.823],
            [3.0,       0.807,     0.401, 0.927, 0,    0,        0.706],
            [3.27,      0.281,     0.879, 0.741, 0,    1,        0.697],
            [3.0,       0.540,     0.410, 0.691, 1,    0,        0.374],
            [2.0,       0.268,     0.544, 0.670, 1,    0,        0.409],
            [4.0,       0.098,     0.907, 0.804, 1,    1,        0.776],
            [3.0,       0.491,     0.867, 0.870, 0,    1,        0.646],
            [2.0,       0.347,     0.548, 0.691, 0,    0,        0.499],
            [3.0,       0.525,     0.576, 0.339, 1,    1,        0.396],
            [2.0,       0.571,     0.460, 0.675, 0,    0,        0.509],
            [2.0,       0.315,     0.523, 0.658, 0,    0,        0.464],
            [3.22,      0.329,     0.726, 0.776, 0,    0,        0.731],
            [2.58,      0.226,     0.661, 0.668, 1,    1,        0.742],
            [3.78,      0.241,     0.782, 0.887, 0,    1,        0.819],
            [2.78,      0.153,     0.817, 0.821, 0,    1,        0.464],
            [3.62,      0.380,     0.855, 0.843, 1,    1,        0.631],
            [2.0,       0.421,     0.549, 0.718, 0,    0,        0.428],
            [3.0,       0.483,     0.815, 0.851, 0,    0,        0.672],
            [2.0,       0.315,     0.476, 0.768, 0,    0,        0.430],
            [2.0,       0.369,     0.388, 0.802, 0,    0,        0.685],
            [2.0,       0.698,     0.437, 0.749, 0,    0,        0.608],
            [3.0,       0.671,     0.447, 0.809, 0,    0,        0.761],
            [3.0,       0.245,     0.835, 0.880, 0,    1,        0.706],
            [2.0,       0.686,     0.413, 0.759, 0,    0,        0.420],
            [3.0,       0.135,     0.839, 0.913, 0,    0,        0.788],
            [3.33,      0.309,     0.774, 0.776, 0,    0,        0.526],
            [3.0,       0.122,     0.909, 0.837, 0,    1,        0.701],
            [3.0,       0.720,     0.494, 0.807, 0,    0,        0.747],
            [2.0,       0.655,     0.389, 0.745, 0,    0,        0.469],
            [1.0,       0.313,     0.436, 0.609, 0,    0,        0.536],
            [3.0,       0.720,     0.380, 0.892, 0,    0,        0.755],
            [2.0,       0.661,     0.512, 0.712, 0,    0,        0.574],
            [3.0,       0.747,     0.453, 0.836, 0,    0,        0.545],
            [3.42,      0.184,     0.762, 0.885, 1,    1,        0.579],
            [2.0,       0.574,     0.595, 0.746, 0,    0,        0.419],
            [3.14,      0.135,     0.839, 0.927, 0,    1,        0.860],
            [3.0,       0.399,     0.869, 0.727, 0,    0,        0.723],
            [3.0,       0.702,     0.311, 0.504, 0,    0,        0.394],
            [3.0,       0.872,     0.365, 0.894, 0,    0,        0.787],
            [3.29,      0.274,     0.837, 0.939, 0,    0,        0.828],
            [3.30,      0.247,     0.789, 0.928, 0,    1,        0.944],
            [4.0,       0.092,     0.845, 0.860, 1,    1,        0.900],
            [3.0,       0.105,     0.850, 0.888, 0,    0,        0.812],
            [4.0,       0.231,     0.955, 0.857, 0,    0,        0.884],
            [4.0,       0.135,     0.850, 0.850, 0,    1,        0.946],
            [3.0,       0.200,     0.871, 0.856, 1,    1,        0.636],
            [3.0,       0.694,     0.462, 0.658, 0,    0,        0.322],
            [3.84,      0.277,     0.740, 0.825, 0,    1,        0.836],
            [3.0,       0.483,     0.411, 0.557, 1,    0,        0.384],
            [3.61,      0.243,     0.866, 0.825, 0,    0,        0.897],
            [3.0,       0.600,     0.520, 0.894, 0,    0,        0.734],
            [2.30,      0.147,     0.873, 0.851, 1,    1,        0.694],
            [3.0,       0.137,     0.758, 0.948, 1,    0,        0.737],
            [3.0,       0.898,     0.400, 0.909, 0,    0,        0.728],
            [3.58,      0.341,     0.735, 0.854, 0,    0,        0.604],
            [3.0,       0.265,     0.814, 0.862, 1,    0,        0.802],
            [2.07,      0.212,     0.653, 0.726, 1,    1,        0.573],
            [2.0,       0.734,     0.405, 0.672, 0,    0,        0.618],
            [4.0,       0.240,     0.836, 0.852, 0,    0,        0.775],
            [1.0,       0.351,     0.566, 0.558, 0,    0,        0.360],
            [1.14,      0.157,     0.470, 0.624, 0,    0,        0.283],
            [3.0,       0.492,     0.512, 0.470, 1,    1,        0.245],
            [3.0,       0.935,     0.514, 0.815, 0,    0,        0.665],
            [2.0,       0.428,     0.370, 0.782, 0,    0,        0.581],
            [3.0,       0.303,     0.581, 0.313, 0,    0,        0.258],
            [2.0,       0.424,     0.514, 0.782, 0,    0,        0.493],
            [2.0,       0.516,     0.550, 0.672, 0,    0,        0.404],
            [1.15,      0.119,     0.371, 0.670, 1,    0,        0.351],
            [4.0,       0.097,     0.884, 0.801, 0,    1,        0.459],
            [2.0,       0.615,     0.326, 0.741, 0,    0,        0.527],
            [3.0,       0.443,     0.373, 0.498, 0,    0,        0.387],
            [2.0,       0.587,     0.477, 0.705, 0,    0,        0.473],
            [3.0,       0.293,     0.562, 0.391, 1,    0,        0.310],
            [3.0,       0.331,     0.550, 0.344, 0,    1,        0.213],
            [1.0,       0.373,     0.459, 0.660, 0,    0,        0.538],
            [3.85,      0.145,     0.754, 0.940, 0,    0,        0.932],
            [3.0,       0.679,     0.386, 0.829, 0,    0,        0.639],
            [2.0,       0.583,     0.598, 0.755, 0,    0,        0.509],
            [3.67,      0.190,     0.779, 0.850, 0,    0,        0.687],
            [3.0,       0.795,     0.512, 0.853, 0,    0,        0.841],
            [3.0,       0.941,     0.424, 0.896, 0,    0,        0.900],
            [3.0,       0.236,     0.867, 0.936, 0,    0,        0.618],
            [3.0,       0.595,     0.447, 0.612, 1,    0,        0.353],
            [3.0,       0.705,     0.527, 0.764, 0,    0,        0.538],
            [3.0,       0.860,     0.352, 0.891, 0,    0,        0.860],
            [3.0,       0.621,     0.502, 0.877, 0,    0,        0.655],
            [2.0,       0.226,     0.539, 0.645, 1,    0,        0.592],
            [3.45,      0.288,     0.854, 0.914, 0,    1,        0.836],
            [2.28,      0.160,     0.786, 0.691, 1,    1,        0.403],
            [3.0,       0.794,     0.329, 0.933, 0,    0,        0.709],
            [2.0,       0.418,     0.547, 0.752, 0,    0,        0.599],
            [3.78,      0.255,     0.881, 0.904, 0,    0,        0.789],
            [2.0,       0.531,     0.426, 0.699, 0,    0,        0.572],
            [3.16,      0.302,     0.858, 0.870, 0,    0,        0.722],
            [3.0,       0.637,     0.371, 0.645, 0,    0,        0.270],
            [3.0,       0.311,     0.799, 0.786, 0,    1,        0.609],
            [2.64,      0.257,     0.827, 0.731, 1,    1,        0.786],
            [3.85,      0.300,     0.724, 0.947, 0,    0,        0.710],
            [3.0,       0.655,     0.366, 0.819, 0,    0,        0.836],
            [1.30,      0.293,     0.356, 0.700, 1,    1,        0.406],
            [3.0,       0.380,     0.788, 0.722, 0,    1,        0.572],
            [3.49,      0.352,     0.851, 0.630, 0,    0,        0.476],
            [3.0,       0.261,     0.838, 0.919, 0,    1,        0.666],
            [3.0,       0.281,     0.909, 0.886, 0,    1,        0.842],
            [3.0,       0.711,     0.422, 0.608, 0,    0,        0.408],
            [3.38,      0.210,     0.814, 0.830, 0,    0,        0.726],
            [3.0,       0.655,     0.327, 0.765, 0,    0,        0.608],
            [3.95,      0.141,     0.804, 0.930, 0,    1,        0.913],
            [4.0,       0.061,     0.878, 0.619, 0,    1,        0.828],
            [3.0,       0.335,     0.853, 0.887, 0,    0,        0.726],
            [2.40,      0.124,     0.682, 0.726, 1,    1,        0.699],
            [3.0,       0.340,     0.556, 0.396, 0,    1,        0.368],
            [3.40,      0.400,     0.717, 0.738, 0,    0,        0.481],
            [2.0,       0.461,     0.394, 0.676, 1,    0,        0.475],
            [3.66,      0.185,     0.866, 0.824, 0,    0,        0.752],
            [3.0,       0.511,     0.458, 0.320, 1,    1,        0.368],
            [2.0,       0.662,     0.387, 0.737, 0,    0,        0.665],
            [3.72,      0.444,     0.902, 0.796, 0,    0,        0.722],
            [4.0,       0.054,     0.934, 0.561, 0,    1,        0.330],
            [3.0,       0.359,     0.710, 0.802, 0,    0,        0.679],
            [2.0,       0.746,     0.471, 0.730, 0,    0,        0.513],
            [4.0,       0.172,     0.976, 0.712, 0,    1,        0.493],
            [2.0,       0.562,     0.558, 0.773, 0,    0,        0.607],
            [1.37,      0.337,     0.353, 0.561, 1,    1,        0.402],
            [3.04,      0.299,     0.769, 0.907, 0,    1,        0.845],
            [1.0,       0.493,     0.426, 0.503, 0,    0,        0.360],
            [3.38,      0.313,     0.887, 0.824, 0,    0,        0.818],
            [3.0,       0.245,     0.848, 0.912, 0,    0,        0.775],
            [3.0,       0.197,     0.889, 0.942, 0,    1,        0.743],
            [3.56,      0.254,     0.860, 0.915, 0,    1,        0.802],
            [1.95,      0.322,     0.358, 0.673, 0,    1,        0.382],
            [3.56,      0.279,     0.738, 0.798, 0,    0,        0.717],
            [3.0,       0.441,     0.493, 0.673, 0,    0,        0.353],
            [3.70,      0.264,     0.837, 0.935, 1,    1,        0.748],
            [3.0,       0.713,     0.360, 0.949, 0,    0,        0.866],

        ])

        #y_train = np.array([2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0])

        y_train = np.array([
        2, 2, 2, 2, 2, 2,  # Original Red alerts (6 samples)
        1, 1, 1, 1, 1,     # Original Yellow advisories (5 samples)
        0, 0, 0, 0, 0,     # Original No alerts (5 samples)

        # --- RED ALERT SAMPLES ---
        2,                  # Tornado 1 min away at max speed
        2,                  # Tornado 4 mins away (Scenario 6 Match!)
        2,                  # Extreme Hail imminent
        2,                  # Tornado 5 mins away on Highway

        # --- YELLOW ADVISORY SAMPLES ---
        1,                  # Heavy Rain, 8 mins away
        1,                  # Tornado 15 mins away (Plenty of time)
        1,                  # Storm ahead, vehicle already braking

        # --- NO ALERT SAMPLES ---
        0,                  # Clear Skies, High Speed
        0,                  # Light Mist, slow residential driving
        0,                  # Light Rain very far away
            # --- APPENDED FROM synthetic_data.txt (label 1→1, label 2→2) ---
        1,1,2,2,2,2,2,2,2,1,1,2,1,1,2,2,1,1,2,2,
        1,1,1,2,1,2,2,2,1,2,2,2,1,2,2,1,2,1,2,1,
        2,1,1,2,1,2,1,1,2,1,1,1,2,2,1,2,1,2,1,2,
        2,2,1,2,1,2,2,1,1,1,2,1,2,1,1,2,2,2,1,1,
        2,1,2,2,1,1,2,1,2,2,2,1,2,1,2,1,2,1,2,2,
        2,2,1,2,1,2,2,2,1,2,2,2,1,2,2,1,2,1,2,1,
        2,1,1,2,1,2,1,1,2,1,1,1,2,2,1,2,1,2,1,2,
        2,2,1,2,1,2,2,1,1,1,2,1,2,1,1,2,2,2,1,1,
        2,1,2,2,1,1,2,1,2,2,2,1,2,1,2,1,2,1,2,2,
        2,2,1,1,1,2,1,2,2,2,1,1,2,1,2,1,2,1,2,1,
        ])

        print(f"Loaded X_train from . Shape: {X_train.shape}")
        print(f"Loaded y_train from . Shape: {y_train.shape}")


    model = RandomForestClassifier(
        n_estimators=100,
        max_depth=5,
        random_state=42,
    )
    model.fit(X_train, y_train)
    return model


# Train the model once at import time
ALERT_MODEL = _build_alert_model()
SEVERITY_NUMERIC = {0: 0, 1: 0, 2: 0, 3: 0, 45: 1, 48: 2,
                    65: 2, 75: 3, 82: 3, 95: 3, 96: 4, 99: 4}


def _build_decision_features(
    pings:        List[GPSPing],
    corridor:     PathCorridor,
    intersection: Optional[WeatherIntersection],
    current_code: int,
) -> np.ndarray:
    """
    Extract the 7-feature vector that feeds into the Random Forest.

    Returns:
        1D numpy array of shape (7,) ready for model.predict_proba().
    """
    # Severity score (0–4)
    sev_code      = intersection.severity_code if intersection else current_code
    severity_num  = SEVERITY_NUMERIC.get(sev_code, 0)

    # Time to impact normalized to [0, 1] (0 = now, 1 = 20 min out)
    tti           = intersection.minutes_to_impact if intersection else 20
    time_norm     = min(1.0, tti / 20)

    # Intersection probability (0–1)
    p_int         = intersection.intersection_prob if intersection else 0.0

    # Road type predictability score (0–1)
    road_info     = ROAD_TYPES.get(corridor.road_type, ROAD_TYPES['unknown'])
    road_score    = road_info['predictability']

    # Is the driver braking? (1 = yes, deceleration > 1 mph/s)
    avg_accel     = sum(p.accel for p in pings[-3:]) / min(3, len(pings))
    is_braking    = 1 if avg_accel < -1.0 else 0

    # Is current location already in severe weather?
    curr_severe   = 1 if current_code in ALERT_THRESHOLD_CODES else 0

    # Speed normalized to [0, 1] (highway speed ~75 mph = 1.0)
    speed_norm    = min(1.0, pings[-1].speed_mph / 75)

    return np.array([severity_num, time_norm, p_int, road_score,
                     is_braking, curr_severe, speed_norm], dtype=float)


print('✅ Phase 4A — Random Forest model trained!')
# Removed redundant print statements as X_train and y_train are local to _build_alert_model()
# print(f"Loaded X_train from . Shape: {X_train.shape}")
# print(f"Loaded y_train from . Shape: {y_train.shape}")

Loaded X_train from /content/sample_data/x_train_biased_large.csv. Shape: (12000, 7)
Loaded y_train from /content/sample_data/y_train_biased_large.csv. Shape: (12000,)
✅ Phase 4A — Random Forest model trained!


In [112]:
# This cell was for Google Drive mounting but is no longer needed.
# Keep it empty or remove if permitted.

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [113]:
# ─────────────────────────────────────────────────────────────────
# PHASE 4B — ALERT DECISION & OUTPUT ASSEMBLY
# ─────────────────────────────────────────────────────────────────

### It is the final assembly line where data from every previous phase—GPS telemetry, path prediction, and weather polygons—is synthesized into a single alert object.

ALERT_THRESHOLDS = {
    'RED':    0.70,  # ≥ 70% → Red Alert
    'YELLOW': 0.30,  # 30–70% → Yellow Advisory
}


def decide_alert(
    pings:           List[GPSPing],
    corridor:        PathCorridor,
    intersection:    Optional[WeatherIntersection],
    weather_response: Optional[Dict],
) -> MovingVehicleAlert:
    """
    Run the Random Forest decision model and assemble the final alert.

    Steps:
      1. Extract 7-feature decision vector
      2. Get probability estimate from Random Forest
      3. Apply Red/Yellow/None thresholds
      4. Select contextual advice and exit tip
      5. Return MovingVehicleAlert

    Args:
        pings:           Phase 1 GPS features.
        corridor:        Phase 2 path corridor.
        intersection:    Phase 3 weather intersection result.
        weather_response: Open-Meteo API response for current conditions.
    Returns:
        Fully populated MovingVehicleAlert.
    """
    # Extract current weather from Open-Meteo response
    current_code  = 0
    wind_speed    = 0.0
    temperature_f = 70.0
    if weather_response:
        curr = weather_response.get('current', {})
        current_code  = curr.get('weather_code', 0)
        wind_speed    = curr.get('wind_speed_10m', 0.0)
        temperature_f = curr.get('temperature_2m', 70.0)

    current_condition = SEVERE_WEATHER_CODES.get(current_code, 'Unknown')

    # ── Random Forest decision ──
    features = _build_decision_features(pings, corridor, intersection, current_code)
    proba    = ALERT_MODEL.predict_proba(features.reshape(1, -1))[0]
    # Classes are [0=None, 1=Yellow, 2=Red] — use P(Red) for primary threshold
    p_red    = proba[2] if len(proba) > 2 else 0.0
    p_yellow = proba[1] if len(proba) > 1 else 0.0
    # Combined impact probability (used for output display)
    p_impact = intersection.intersection_prob if intersection else 0.0

    # ── Apply thresholds ──
    if p_red >= ALERT_THRESHOLDS['RED']:
        alert_level = 'RED'
    elif (p_red + p_yellow) >= ALERT_THRESHOLDS['YELLOW']:
        alert_level = 'YELLOW'
    elif current_code in ALERT_THRESHOLD_CODES:
        # Current position is already in severe weather — always alert
        alert_level = 'RED' if ALERT_SEVERITY.get(current_code) in ('HIGH', 'CRITICAL') else 'YELLOW'
    else:
        alert_level = 'NONE'

    # ── Build contextual messages ──
    advice_code    = (intersection.severity_code if intersection else current_code)
    vehicle_advice = MOVING_VEHICLE_ADVICE.get(advice_code, '')
    exit_tip       = intersection.exit_before_impact if intersection else None

    if alert_level == 'RED':
        wx_label = intersection.weather_type if intersection else current_condition
        tti_str  = (f'in ~{int(intersection.minutes_to_impact)} min'
                    if intersection and intersection.minutes_to_impact > 0
                    else 'at your location NOW')
        alert_message = (
            f'🔴 RED ALERT [{ALERT_SEVERITY.get(advice_code, "HIGH")}]: '
            f'{wx_label} on your predicted path {tti_str}. '
            f'Impact probability: {p_impact*100:.0f}%. '
            f'Speed: {pings[-1].speed_mph:.0f} mph | '
            f'Heading: {pings[-1].heading:.0f}°'
        )
    elif alert_level == 'YELLOW':
        wx_label = intersection.weather_type if intersection else current_condition
        tti_str  = (f'in ~{int(intersection.minutes_to_impact)} min'
                    if intersection and intersection.minutes_to_impact > 0
                    else 'near your location')
        alert_message = (
            f'🟡 YELLOW ADVISORY: {wx_label} possible on your path {tti_str}. '
            f'Impact probability: {p_impact*100:.0f}%. Monitor conditions.'
        )
    else:
        alert_message = '✅ No alert needed. Your predicted path is clear of severe weather.'

    return MovingVehicleAlert(
        current_lat      = pings[-1].lat,
        current_lon      = pings[-1].lon,
        speed_mph        = round(pings[-1].speed_mph, 1),
        heading_degrees  = round(pings[-1].heading, 1),
        road_type        = corridor.road_type,
        corridor         = corridor,
        intersection     = intersection,
        current_weather  = current_condition,
        current_code     = current_code,
        wind_speed_mph   = wind_speed,
        temperature_f    = temperature_f,
        alert_level      = alert_level,
        alert_message    = alert_message,
        vehicle_advice   = vehicle_advice,
        exit_tip         = exit_tip,
        confidence_pct   = round(p_impact * 100, 1),
    )


def run_phase4(
    pings:           List[GPSPing],
    corridor:        PathCorridor,
    intersection:    Optional[WeatherIntersection],
    weather_response: Optional[Dict],
) -> MovingVehicleAlert:
    """Full Phase 4 pipeline: decision features → RF model → alert assembly."""
    return decide_alert(pings, corridor, intersection, weather_response)


print('✅ Phase 4B — Alert decision engine defined!')

✅ Phase 4B — Alert decision engine defined!


## 🔗 Full Pipeline & Display

In [114]:
# ─────────────────────────────────────────────────────────────────
# FULL PIPELINE ORCHESTRATOR
# ─────────────────────────────────────────────────────────────────

def run_safe_pilot_moving(
    raw_pings: List[GPSPing],
    label: str = 'Vehicle',
    _mock_weather: Optional[Dict] = None,
    _mock_nws: Optional[List] = None,
) -> MovingVehicleAlert:
    """
    Main entry point: runs all 4 phases for a moving vehicle.

    Args:
        raw_pings:     List of 10–15 GPSPing objects (oldest → newest).
        label:         Human-readable vehicle/scenario name for display.
        _mock_weather: Optional injected Open-Meteo response (for simulation).
        _mock_nws:     Optional injected NWS polygon list (for simulation).
    Returns:
        MovingVehicleAlert — the fully-reasoned alert for this vehicle state.
    """
    print(f'\n🛣️  Running Safe Pilot for: {label}')
    print(f'   GPS pings: {len(raw_pings)} | '
          f'Lat: {raw_pings[-1].lat:.4f} | Lon: {raw_pings[-1].lon:.4f}')

    # Phase 1 — Feature engineering
    print('   ⚙️  Phase 1: Feature extraction...')
    pings = run_phase1(deepcopy(raw_pings))

    # Phase 2 — Trajectory prediction
    print('   🧠 Phase 2: LSTM trajectory prediction...')
    corridor = run_phase2(pings)

    # Phase 3 — Weather intersection
    print('   🌩️  Phase 3: Weather intersection...')
    if _mock_weather is not None or _mock_nws is not None:
        # Simulation mode: use injected data
        nws_polygons     = _mock_nws if _mock_nws is not None else []
        weather_response = _mock_weather
        intersection     = intersect_corridor_with_weather(
            corridor, nws_polygons, weather_response, pings
        )
    else:
        intersection, weather_response = run_phase3(pings, corridor)

    # Phase 4 — Alert decision
    print('   🎯 Phase 4: Alert decision...')
    alert = run_phase4(pings, corridor, intersection, weather_response)

    return alert


def print_moving_alert(alert: MovingVehicleAlert, label: str = ''):
    """
    Display a formatted MovingVehicleAlert — designed for glanceable in-car UX.
    High-contrast layout with the most critical info at the top.

    Args:
        alert: Populated MovingVehicleAlert from run_safe_pilot_moving().
        label: Optional scenario name for the header.
    """
    SEP  = '=' * 72
    DASH = '-' * 72

    print(SEP)
    if label:
        print(f'  🛡️  {label}')
    print(f'  📍 Position : {alert.current_lat:.4f}°N, {alert.current_lon:.4f}°W')
    print(f'  🚗 Speed    : {alert.speed_mph:.0f} mph  |  '
          f'Heading: {alert.heading_degrees:.0f}°  |  '
          f'Road: {alert.road_type}')
    print(DASH)

    # Current conditions
    print(f'  🌡️  Temp     : {alert.temperature_f:.1f} °F')
    print(f'  🌤️  Now      : {alert.current_weather} (code {alert.current_code})')
    print(f'  💨 Wind     : {alert.wind_speed_mph:.1f} mph')

    # Predicted path corridor
    print(DASH)
    print(f'  🧠 Predicted Path Corridor (next 20 min):')
    for wp in alert.corridor.waypoints:
        print(f'     +{wp.minutes_ahead:2d} min → '
              f'({wp.lat:.4f}°, {wp.lon:.4f}°)  '
              f'conf={wp.confidence:.2f}  '
              f'width=±{wp.corridor_radius_m:.0f}m')
    print(f'  📊 Corridor avg confidence: {alert.corridor.avg_confidence:.2f}')

    # Weather intersection
    print(DASH)
    if alert.intersection:
        ix = alert.intersection
        print(f'  ⚡ Weather on Path: {ix.weather_type}')
        print(f'     Time to Impact : ~{ix.minutes_to_impact:.0f} min')
        print(f'     P(Impact)      : {ix.intersection_prob*100:.0f}%')
    else:
        print('  ✅  No weather threats detected on predicted path.')

    # Alert decision
    print(DASH)
    alert_icon = {'RED': '🔴', 'YELLOW': '🟡', 'NONE': '✅'}.get(alert.alert_level, '❓')
    print(f'  {alert_icon} {alert.alert_message}')

    if alert.vehicle_advice:
        print(f'\n  💡 Driver Action: {alert.vehicle_advice}')

    if alert.exit_tip:
        print(f'  🛣️  Exit Guidance : {alert.exit_tip}')

    print(SEP)
    print()


print('✅ Pipeline orchestrator & display defined!')

✅ Pipeline orchestrator & display defined!


## 🧰 Simulation Helpers

In [115]:
# ─────────────────────────────────────────────────────────────────
# SIMULATION HELPERS
# ─────────────────────────────────────────────────────────────────

### Since we can't always wait for a real tornado to happen just to test if our code works, we use Simulation Helpers.
#Think of these as a "Flight Simulator" for the Safe Pilot system. They allow us to create "Digital Twins" of cars and storms to verify the logic.
#no actual live data is being passed yet—these are the blueprints for creating data.
#However, these functions are designed to generate the "Sample Data" that the rest of the system needs to run

def make_gps_trace(
    start_lat: float, start_lon: float,
    heading_deg: float, speed_mph: float,
    n_pings: int = 12, interval_sec: int = 5,
) -> List[GPSPing]:
    """
    Generate a synthetic GPS trace for a vehicle moving at constant
    speed and heading.  Adds small Gaussian noise to simulate real GPS jitter.

    Args:
        start_lat/lon:  Starting coordinate.
        heading_deg:    Direction of travel (0–360°, clockwise from North).
        speed_mph:      Speed of travel.
        n_pings:        Number of GPS samples to generate.
        interval_sec:   Seconds between each GPS ping.
    Returns:
        List of GPSPing objects ready for Phase 1 processing.
    """
    speed_ms   = speed_mph / MPH_PER_MS  # Convert to m/s
    dist_per_ping = speed_ms * interval_sec  # Metres per ping
    base_time  = datetime.utcnow() - timedelta(seconds=n_pings * interval_sec)

    pings = []
    lat, lon = start_lat, start_lon

    for i in range(n_pings):
        # Add GPS jitter (±5 m laterally, ±3 m along path)
        jitter_lat = np.random.normal(0, 0.00004)  # ~4 m lat jitter
        jitter_lon = np.random.normal(0, 0.00004)  # ~4 m lon jitter

        pings.append(GPSPing(
            lat=lat + jitter_lat,
            lon=lon + jitter_lon,
            alt=200.0 + np.random.normal(0, 1),
            timestamp=base_time + timedelta(seconds=i * interval_sec),
        ))

        # Advance position for next ping
        lat, lon = project_point(lat, lon, heading_deg, dist_per_ping)

    return pings


def make_mock_nws_polygon(
    center_lat: float, center_lon: float,
    radius_deg: float,
    event: str = 'Severe Thunderstorm Warning',
) -> List[Dict]:
    """
    Create a synthetic NWS warning polygon centered at a coordinate.
    Used in simulated scenarios to place a weather zone on the vehicle's path.

    Args:
        center_lat/lon: Center of the weather polygon.
        radius_deg:     Radius in decimal degrees (~0.1° ≈ 7 miles).
        event:          NWS event type string.
    Returns:
        List with one mock NWS polygon dict.
    """
    n_pts = 16
    pts = [
        (
            center_lon + radius_deg * math.cos(2 * math.pi * i / n_pts),
            center_lat + radius_deg * math.sin(2 * math.pi * i / n_pts),
        )
        for i in range(n_pts)
    ]
    polygon = Polygon(pts)
    code = NWS_EVENT_TO_CODE.get(event, 95)
    return [{'event': event, 'severity': 'Severe', 'polygon': polygon, 'code': code}]


def make_mock_weather_response(
    current_code: int = 0,
    wind_mph: float = 10.0,
    temp_f: float = 75.0,
    hourly_codes: Optional[List[int]] = None,
) -> Dict:
    """
    Build a minimal mock Open-Meteo API response for simulation.
    Injects specific weather codes without making a real API call.
    """
    codes = hourly_codes if hourly_codes else [current_code] * 24
    now   = datetime.utcnow()
    times = [(now + timedelta(hours=i)).strftime('%Y-%m-%dT%H:00') for i in range(24)]
    return {
        'current': {
            'weather_code':   current_code,
            'wind_speed_10m': wind_mph,
            'temperature_2m': temp_f,
            'precipitation':  0.0,
            'visibility':     24000,
        },
        'hourly': {
            'time':                     times,
            'weather_code':             codes,
            'precipitation_probability': [60 if c in ALERT_THRESHOLD_CODES else 5 for c in codes],
            'wind_gusts_10m':           [65.0 if c in ALERT_THRESHOLD_CODES else 10.0 for c in codes],
            'visibility':               [2000 if c in ALERT_THRESHOLD_CODES else 24000 for c in codes],
        },
    }


print('✅ Simulation helpers defined!')

✅ Simulation helpers defined!


In [116]:
np.random.seed(42)  # Reproducible GPS jitter

print('\n' + '🛡️  SAFE PILOT — SECTION A: Live Moving Vehicle Demo'.center(72))
print(f"{'Run at: ' + datetime.now().strftime('%Y-%m-%d %H:%M:%S'):^72}\n")

# ── Live scenario definitions ──
LIVE_SCENARIOS = [
    {
        'label':       'Vehicle 1 — Oklahoma City, OK (Tornado Alley, Northbound I-35)',
        'start_lat':   35.4676, 'start_lon': -97.5164,
        'heading':     0,       # Northbound
        'speed_mph':   70,
    },
    {
        'label':       'Vehicle 2 — Miami, FL (Eastbound I-836, Hurricane Season)',
        'start_lat':   25.7617, 'start_lon': -80.3000,
        'heading':     90,      # Eastbound
        'speed_mph':   55,
    },
    {
        'label':       'Vehicle 3 — Denver, CO (Southbound I-25, Mountain Weather)',
        'start_lat':   39.7392, 'start_lon': -104.9903,
        'heading':     180,     # Southbound
        'speed_mph':   65,
    },
]

live_alerts = []
for scenario in LIVE_SCENARIOS:
    pings = make_gps_trace(
        start_lat   = scenario['start_lat'],
        start_lon   = scenario['start_lon'],
        heading_deg = scenario['heading'],
        speed_mph   = scenario['speed_mph'],
    )
    alert = run_safe_pilot_moving(pings, label=scenario['label'])
    print_moving_alert(alert, label=scenario['label'])
    live_alerts.append((scenario['label'], alert))

# Section A Summary
print('-' * 72)
print('📊 SECTION A SUMMARY')
print('-' * 72)
for label, alert in live_alerts:
    icon = {'RED': '🔴', 'YELLOW': '🟡', 'NONE': '✅'}.get(alert.alert_level, '❓')
    print(f'  {icon} {label}')
print('-' * 72)


          🛡️  SAFE PILOT — SECTION A: Live Moving Vehicle Demo          
                      Run at: 2026-04-14 16:27:18                       


🛣️  Running Safe Pilot for: Vehicle 1 — Oklahoma City, OK (Tornado Alley, Northbound I-35)
   GPS pings: 12 | Lat: 35.4830 | Lon: -97.5164
   ⚙️  Phase 1: Feature extraction...
   🧠 Phase 2: LSTM trajectory prediction...
   🌩️  Phase 3: Weather intersection...
  ⚠️  NWS API call failed: 400 Client Error: Bad Request for url: https://api.weather.gov/alerts/active?point=35.4823%2C-97.5163&status=actual&limit=50 — continuing without NWS polygons.


/tmp/ipykernel_1742/3376294222.py:30: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  base_time  = datetime.utcnow() - timedelta(seconds=n_pings * interval_sec)


   🎯 Phase 4: Alert decision...
  🛡️  Vehicle 1 — Oklahoma City, OK (Tornado Alley, Northbound I-35)
  📍 Position : 35.4823°N, -97.5163°W
  🚗 Speed    : 34 mph  |  Heading: 2°  |  Road: primary
------------------------------------------------------------------------
  🌡️  Temp     : 80.1 °F
  🌤️  Now      : Mainly Clear (code 1)
  💨 Wind     : 18.0 mph
------------------------------------------------------------------------
  🧠 Predicted Path Corridor (next 20 min):
     + 5 min → (35.5582°, -97.5149°)  conf=0.10  width=±735m
     +10 min → (35.6341°, -97.5135°)  conf=0.10  width=±735m
     +15 min → (35.7099°, -97.5121°)  conf=0.10  width=±735m
     +20 min → (35.7858°, -97.5107°)  conf=0.10  width=±735m
  📊 Corridor avg confidence: 0.10
------------------------------------------------------------------------
  ⚡ Weather on Path: Thunderstorm
     Time to Impact : ~5 min
     P(Impact)      : 6%
------------------------------------------------------------------------
  🟡 🟡 YELLOW ADVI

/tmp/ipykernel_1742/3376294222.py:30: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  base_time  = datetime.utcnow() - timedelta(seconds=n_pings * interval_sec)


  ⚠️  NWS API call failed: 400 Client Error: Bad Request for url: https://api.weather.gov/alerts/active?point=25.7617%2C-80.2871&status=actual&limit=50 — continuing without NWS polygons.
   🎯 Phase 4: Alert decision...
  🛡️  Vehicle 2 — Miami, FL (Eastbound I-836, Hurricane Season)
  📍 Position : 25.7617°N, -80.2871°W
  🚗 Speed    : 27 mph  |  Heading: 89°  |  Road: secondary
------------------------------------------------------------------------
  🌡️  Temp     : 80.0 °F
  🌤️  Now      : Mainly Clear (code 1)
  💨 Wind     : 11.8 mph
------------------------------------------------------------------------
  🧠 Predicted Path Corridor (next 20 min):
     + 5 min → (25.7618°, -80.2206°)  conf=0.43  width=±518m
     +10 min → (25.7619°, -80.1540°)  conf=0.36  width=±569m
     +15 min → (25.7619°, -80.0875°)  conf=0.28  width=±620m
     +20 min → (25.7619°, -80.0210°)  conf=0.20  width=±671m
  📊 Corridor avg confidence: 0.32
------------------------------------------------------------------

/tmp/ipykernel_1742/3376294222.py:30: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  base_time  = datetime.utcnow() - timedelta(seconds=n_pings * interval_sec)


   🎯 Phase 4: Alert decision...
  🛡️  Vehicle 3 — Denver, CO (Southbound I-25, Mountain Weather)
  📍 Position : 39.7255°N, -104.9903°W
  🚗 Speed    : 32 mph  |  Heading: 178°  |  Road: primary
------------------------------------------------------------------------
  🌡️  Temp     : 57.2 °F
  🌤️  Now      : Overcast (code 3)
  💨 Wind     : 8.0 mph
------------------------------------------------------------------------
  🧠 Predicted Path Corridor (next 20 min):
     + 5 min → (39.6549°, -104.9894°)  conf=0.10  width=±735m
     +10 min → (39.5844°, -104.9885°)  conf=0.10  width=±735m
     +15 min → (39.5138°, -104.9876°)  conf=0.10  width=±735m
     +20 min → (39.4432°, -104.9868°)  conf=0.10  width=±735m
  📊 Corridor avg confidence: 0.10
------------------------------------------------------------------------
  ✅  No weather threats detected on predicted path.
------------------------------------------------------------------------
  ✅ ✅ No alert needed. Your predicted path is clear of 

In [117]:
np.random.seed(42)

print('\n' + '🧪  SAFE PILOT — SECTION B: Simulated Moving Scenarios'.center(72) + '\n')

# Base location: Edinburg, TX (warm, clear baseline)
BASE_LAT, BASE_LON = 26.3017, -98.1633

SIM_SCENARIOS = [
    # ── Scenario 1: Tornado Warning polygon placed ~8 miles ahead on highway ──
    {
        'label':       'Scenario 1 | 70 mph Highway → Tornado Warning in Path [RED]',
        'heading':     0, 'speed_mph': 70,
        'mock_weather': make_mock_weather_response(current_code=2, wind_mph=25, temp_f=82),
        'mock_nws':    make_mock_nws_polygon(
            center_lat=BASE_LAT + 0.18,  # ~12 miles north = within 20 min at 70 mph
            center_lon=BASE_LON,
            radius_deg=0.12,
            event='Tornado Warning',
        ),
    },
    # ── Scenario 2: Severe Thunderstorm Warning directly on predicted path ──
    {
        'label':       'Scenario 2 | 65 mph Highway → Thunderstorm + Hail Ahead [RED]',
        'heading':     45, 'speed_mph': 65,
        'mock_weather': make_mock_weather_response(current_code=3, wind_mph=18, temp_f=78),
        'mock_nws':    make_mock_nws_polygon(
            center_lat=BASE_LAT + 0.12,
            center_lon=BASE_LON + 0.12,
            radius_deg=0.10,
            event='Severe Thunderstorm Warning',
        ),
    },
    # ── Scenario 3: Flash Flood on residential road, driver braking ──
    {
        'label':       'Scenario 3 | 20 mph Residential → Flash Flood Zone [YELLOW]',
        'heading':     90, 'speed_mph': 20,
        'mock_weather': make_mock_weather_response(current_code=63, wind_mph=12, temp_f=72),
        'mock_nws':    make_mock_nws_polygon(
            center_lat=BASE_LAT,
            center_lon=BASE_LON + 0.08,  # ~5 miles east = within range but low prob
            radius_deg=0.06,
            event='Flash Flood Warning',
        ),
    },
    # ── Scenario 4: Open-Meteo forecasts heavy rain in 15 min ahead ──
    {
        'label':       'Scenario 4 | 40 mph Primary → Heavy Rain in 15 min [YELLOW]',
        'heading':     270, 'speed_mph': 40,
        'mock_weather': make_mock_weather_response(
            current_code=3, wind_mph=15, temp_f=68,
            hourly_codes=[3, 3, 65, 65, 65, 63, 1] + [0]*17,
        ),
        'mock_nws': [],  # No active NWS polygon
    },
    # ── Scenario 5: Clear sky, no threats anywhere ──
    {
        'label':       'Scenario 5 | 70 mph Highway → Clear Sky, No Alerts [NONE]',
        'heading':     180, 'speed_mph': 70,
        'mock_weather': make_mock_weather_response(current_code=0, wind_mph=8, temp_f=75),
        'mock_nws':    [],
    },
    # ── Scenario 6: CRITICAL Tornado Impact (Forced Red Alert) ──
    {
        'label':       'Scenario 6 | 75 mph Highway → TORNADO IMMINENT [RED]',
        'heading':     0, 'speed_mph': 75,
        'mock_weather': make_mock_weather_response(
            current_code=96,   # Severe Thunderstorm with Hail
            wind_mph=65,
            temp_f=85
        ),
        'mock_nws':    make_mock_nws_polygon(
            center_lat=BASE_LAT + 0.07,  # ~4.8 miles ahead (Impact in < 4 mins)
            center_lon=BASE_LON,
            radius_deg=0.10,
            event='Tornado Warning',
        ),
    },
]

### Simulates GPS pings, n_ping defaulted to 12 pings and interval_sec defaulted to 5 sec

sim_results = []
for s in SIM_SCENARIOS:
    pings = make_gps_trace(
        start_lat=BASE_LAT, start_lon=BASE_LON,
        heading_deg=s['heading'], speed_mph=s['speed_mph'],
    )
    alert = run_safe_pilot_moving(
        pings,
        label=s['label'],
        _mock_weather=s['mock_weather'],
        _mock_nws=s['mock_nws'],
    )
    print_moving_alert(alert, label=s['label'])
    sim_results.append((s['label'], alert))

# Section B Summary
print('-' * 72)
print('📊 SECTION B SUMMARY')
print('-' * 72)
for label, alert in sim_results:
    icon = {'RED': '🔴', 'YELLOW': '🟡', 'NONE': '✅'}.get(alert.alert_level, '❓')
    print(f'  {icon} [{alert.alert_level:6s}] {label}')
print('-' * 72)


         🧪  SAFE PILOT — SECTION B: Simulated Moving Scenarios          


🛣️  Running Safe Pilot for: Scenario 1 | 70 mph Highway → Tornado Warning in Path [RED]
   GPS pings: 12 | Lat: 26.3171 | Lon: -98.1633
   ⚙️  Phase 1: Feature extraction...
   🧠 Phase 2: LSTM trajectory prediction...
   🌩️  Phase 3: Weather intersection...
   🎯 Phase 4: Alert decision...
  🛡️  Scenario 1 | 70 mph Highway → Tornado Warning in Path [RED]
  📍 Position : 26.3164°N, -98.1632°W
  🚗 Speed    : 34 mph  |  Heading: 2°  |  Road: primary
------------------------------------------------------------------------
  🌡️  Temp     : 82.0 °F
  🌤️  Now      : Partly Cloudy (code 2)
  💨 Wind     : 25.0 mph
------------------------------------------------------------------------
  🧠 Predicted Path Corridor (next 20 min):
     + 5 min → (26.3923°, -98.1618°)  conf=0.10  width=±735m
     +10 min → (26.4682°, -98.1604°)  conf=0.10  width=±735m
     +15 min → (26.5440°, -98.1590°)  conf=0.10  width=±735m
     +20 min →

/tmp/ipykernel_1742/3376294222.py:93: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now   = datetime.utcnow()
/tmp/ipykernel_1742/3376294222.py:30: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  base_time  = datetime.utcnow() - timedelta(seconds=n_pings * interval_sec)
/tmp/ipykernel_1742/3376294222.py:30: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  base_time  = datetime.utcnow() - timedelta(seconds=n_pings * interval_sec)
/tmp/ipykernel_1742/3376294222.py:30: DeprecationWarning: datetime.datetime.utcnow() is depreca

In [118]:
np.random.seed(0)

# ── Configure your vehicle state here ──
MY_LAT       = 32.7157   # Replace: your current latitude  (e.g. 40.7128 for NYC)
MY_LON       = -117.1611 # Replace: your current longitude (e.g. -74.0060 for NYC)
MY_HEADING   = 45        # Replace: direction of travel in degrees (0=N, 90=E, 180=S, 270=W)
MY_SPEED_MPH = 65        # Replace: your current speed in mph
LIVE_MODE    = True      # True = real NWS + Open-Meteo APIs; False = no weather APIs

# Generate synthetic GPS trace for your location
my_pings = make_gps_trace(
    start_lat   = MY_LAT,
    start_lon   = MY_LON,
    heading_deg = MY_HEADING,
    speed_mph   = MY_SPEED_MPH,
    n_pings     = 12,
)

if LIVE_MODE:
    # Real APIs — pulls live NWS alerts and Open-Meteo weather
    my_alert = run_safe_pilot_moving(
        my_pings,
        label='My Custom Vehicle',
    )
else:
    # Offline mode — inject a clear sky response
    my_alert = run_safe_pilot_moving(
        my_pings,
        label='My Custom Vehicle (Offline)',
        _mock_weather=make_mock_weather_response(current_code=1, wind_mph=10, temp_f=72),
        _mock_nws=[],
    )

print_moving_alert(my_alert, label='My Custom Vehicle')
my_alert  # Also return the object for inspection


🛣️  Running Safe Pilot for: My Custom Vehicle
   GPS pings: 12 | Lat: 32.7258 | Lon: -117.1490
   ⚙️  Phase 1: Feature extraction...
   🧠 Phase 2: LSTM trajectory prediction...
   🌩️  Phase 3: Weather intersection...
  ⚠️  NWS API call failed: 400 Client Error: Bad Request for url: https://api.weather.gov/alerts/active?point=32.7254%2C-117.1496&status=actual&limit=50 — continuing without NWS polygons.


/tmp/ipykernel_1742/3376294222.py:30: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  base_time  = datetime.utcnow() - timedelta(seconds=n_pings * interval_sec)


   🎯 Phase 4: Alert decision...
  🛡️  My Custom Vehicle
  📍 Position : 32.7254°N, -117.1496°W
  🚗 Speed    : 32 mph  |  Heading: 44°  |  Road: primary
------------------------------------------------------------------------
  🌡️  Temp     : 62.0 °F
  🌤️  Now      : Mainly Clear (code 1)
  💨 Wind     : 1.4 mph
------------------------------------------------------------------------
  🧠 Predicted Path Corridor (next 20 min):
     + 5 min → (32.7745°, -117.0903°)  conf=0.13  width=±716m
     +10 min → (32.8236°, -117.0310°)  conf=0.10  width=±735m
     +15 min → (32.8727°, -116.9716°)  conf=0.10  width=±735m
     +20 min → (32.9217°, -116.9122°)  conf=0.10  width=±735m
  📊 Corridor avg confidence: 0.11
------------------------------------------------------------------------
  ✅  No weather threats detected on predicted path.
------------------------------------------------------------------------
  ✅ ✅ No alert needed. Your predicted path is clear of severe weather.



MovingVehicleAlert(current_lat=32.725364474079, current_lon=-117.1495685531442, speed_mph=31.6, heading_degrees=44.5, road_type='primary', corridor=PathCorridor(waypoints=[PredictedWaypoint(lat=32.774501572865546, lon=-117.0903192051033, minutes_ahead=5, confidence=0.129, corridor_radius_m=716.3), PredictedWaypoint(lat=32.82361075354316, lon=-117.03100441068406, minutes_ahead=10, confidence=0.1, corridor_radius_m=735.0), PredictedWaypoint(lat=32.87269193259311, lon=-116.97162401146178, minutes_ahead=15, confidence=0.1, corridor_radius_m=735.0), PredictedWaypoint(lat=32.92174502627723, lon=-116.91217784883294, minutes_ahead=20, confidence=0.1, corridor_radius_m=735.0)], polygon=<MULTIPOLYGON (((-117.148 32.725, -117.148 32.725, -117.148 32.724, -117.149...>, road_type='primary', avg_confidence=0.107), intersection=None, current_weather='Mainly Clear', current_code=1, wind_speed_mph=1.4, temperature_f=62.0, alert_level='NONE', alert_message='✅ No alert needed. Your predicted path is clea